# TOBi Routing & Escalation Optimization - Analysis Notebook

Companion to the SQL in `models/` and `analysis/`. It connects to BigQuery,
pulls the enriched session-level view, and produces the visuals that back the
five deliverables (misrouting patterns, impact, root cause, recommendations).

**Prerequisites**
1. Build the views first by running `models/01` -> `02` -> `03` -> `04` in BigQuery.
2. Have `google-cloud-bigquery`, `pandas`, `matplotlib`, `seaborn`, `plotly` installed.
3. Authenticated to the GCP project (e.g. `gcloud auth application-default login`).

## 0. Setup

In [ ]:
# If needed, install dependencies (uncomment):
# %pip install google-cloud-bigquery pandas db-dtypes matplotlib seaborn plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

In [ ]:
# ---- CONFIG -------------------------------------------------------------
PROJECT       = 'vf-pt-copsvertex-live'
SOURCE_DS     = 'vfpt_dh_lake_cops_pub_investigation'   # raw tables
ANALYSIS_DS   = 'tobi_routing_analysis'                 # where the views live

# Optional analysis window (set to None to use all data)
DATE_FROM = None   # e.g. '2025-01-01'
DATE_TO   = None   # e.g. '2025-03-31'

client = bigquery.Client(project=PROJECT)

def run_query(sql: str) -> pd.DataFrame:
    """Run a BigQuery SQL string and return a DataFrame."""
    return client.query(sql).to_dataframe()

MASTER = f'`{PROJECT}.{ANALYSIS_DS}.v_session_master`'
print('Reading from', MASTER)

## 1. Load the session master

One row per session with the misrouting / FCR / repeat-contact flags already
derived in `models/04_session_master.sql`.

In [ ]:
where = ''
if DATE_FROM and DATE_TO:
    where = f"WHERE DATE(START_MOMENT) BETWEEN '{DATE_FROM}' AND '{DATE_TO}'"

df = run_query(f'''
SELECT
  SESSION_ID, START_MOMENT, END_MOMENT, CHANNEL, DNIS, FIRST_INTENT,
  confidence_band, technical_topic_type, is_technical_topic,
  routed_queue_category, routed_queue_subtype, final_transfer_target,
  was_transferred, n_transfers, duration_seconds,
  is_hard_misroute, is_soft_misroute, is_correct_technical_route,
  is_fcr, has_next_session, repeat_contact_24h, is_functional_flag
FROM {MASTER}
{where}
''')

print(df.shape)
df.head()

In [ ]:
# Derive a single cohort label for plotting
def cohort(r):
    if r['is_hard_misroute']: return 'misrouted_hard'
    if r['is_soft_misroute']: return 'misrouted_soft'
    if r['is_correct_technical_route']: return 'correct_technical'
    if r['is_technical_topic']: return 'technical_other'
    return 'non_technical'

df['cohort'] = df.apply(cohort, axis=1)
df['cohort'].value_counts()

## 2. Headline KPIs (Deliverables 1 & 2)

In [ ]:
tech = df[df['is_technical_topic']]
kpis = {
    'sessions_total': len(df),
    'technical_sessions': len(tech),
    'hard_misroute_%': round(100*tech['is_hard_misroute'].mean(), 2),
    'any_misroute_%': round(100*(tech['is_hard_misroute'] | tech['is_soft_misroute']).mean(), 2),
    'technical_FCR_%': round(100*tech['is_fcr'].mean(), 2),
    'technical_repeat24h_%': round(100*tech['repeat_contact_24h'].mean(), 2),
    'avg_handovers_technical': round(tech['n_transfers'].mean(), 2),
}
pd.Series(kpis).to_frame('value')

## 3. Misrouting by technical topic (Deliverable 1)

In [ ]:
g = (tech.groupby('technical_topic_type')
         .agg(sessions=('SESSION_ID','count'),
              hard_misroutes=('is_hard_misroute','sum'))
         .assign(pct=lambda d: 100*d['hard_misroutes']/d['sessions'])
         .sort_values('hard_misroutes', ascending=False))

fig, ax = plt.subplots(1, 2, figsize=(13,4))
g['hard_misroutes'].plot.bar(ax=ax[0], color='#c0392b')
ax[0].set_title('Hard misroutes by technical topic'); ax[0].set_ylabel('sessions')
g['pct'].plot.bar(ax=ax[1], color='#e67e22')
ax[1].set_title('Misroute rate (%) by technical topic'); ax[1].set_ylabel('%')
plt.tight_layout(); plt.show()
g

## 4. Root cause: entry point & intent-detection confidence (Deliverable 3)

In [ ]:
def misroute_rate_by(col, min_n=30):
    d = (tech.groupby(col)
             .agg(technical=('SESSION_ID','count'),
                  misroutes=('is_hard_misroute','sum')))
    d = d[d['technical'] >= min_n]
    d['pct_misroute'] = 100*d['misroutes']/d['technical']
    return d.sort_values('pct_misroute', ascending=False)

by_channel = misroute_rate_by('CHANNEL', min_n=1)
by_conf    = misroute_rate_by('confidence_band', min_n=1)

fig, ax = plt.subplots(1, 2, figsize=(13,4))
by_channel['pct_misroute'].plot.bar(ax=ax[0], color='#2980b9')
ax[0].set_title('Misroute % by channel')
by_conf['pct_misroute'].plot.bar(ax=ax[1], color='#8e44ad')
ax[1].set_title('Misroute % by confidence level')
plt.tight_layout(); plt.show()
display(by_channel.head(15)); display(by_conf)

## 5. Impact: resolution time, handovers, repeat contacts (Deliverable 2)

In [ ]:
order = ['non_technical','technical_other','correct_technical','misrouted_soft','misrouted_hard']
plot_df = df[df['duration_seconds'].between(0, df['duration_seconds'].quantile(0.99))]

fig, ax = plt.subplots(1, 2, figsize=(14,5))
sns.boxplot(data=plot_df, x='cohort', y='duration_seconds', order=order, ax=ax[0])
ax[0].set_title('Session duration by cohort (<=p99)'); ax[0].tick_params(axis='x', rotation=30)

summary = (df.groupby('cohort')
             .agg(sessions=('SESSION_ID','count'),
                  avg_handovers=('n_transfers','mean'),
                  pct_repeat=('repeat_contact_24h','mean'))
             .reindex(order))
summary['pct_repeat'] *= 100
summary[['avg_handovers']].plot.bar(ax=ax[1], legend=False, color='#16a085')
ax[1].set_title('Avg handovers by cohort'); ax[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()
summary

## 6. Trend over time

In [ ]:
ts = df.copy()
ts['day'] = pd.to_datetime(ts['START_MOMENT']).dt.date
daily = (ts.groupby('day')
           .agg(technical=('is_technical_topic','sum'),
                misroutes=('is_hard_misroute','sum')))
daily['pct_misroute'] = 100*daily['misroutes']/daily['technical'].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(13,4))
ax.plot(daily.index, daily['pct_misroute'], marker='o', color='#c0392b')
ax.set_title('Daily technical hard-misroute rate (%)'); ax.set_ylabel('%')
plt.tight_layout(); plt.show()
daily.tail(14)

## 7. Conversation-flow Sankey (Deliverable 3)

Intent -> final destination for technical sessions. Red links land in a
non-technical queue (misroutes).

In [ ]:
edges = run_query(f'''
SELECT FIRST_INTENT AS source_intent, final_transfer_target AS target_queue,
       routed_queue_category, COUNT(*) AS n_sessions
FROM {MASTER}
WHERE is_technical_topic AND final_transfer_target IS NOT NULL
GROUP BY 1,2,3 ORDER BY n_sessions DESC LIMIT 40
''')

try:
    import plotly.graph_objects as go
    labels = pd.unique(edges[['source_intent','target_queue']].values.ravel()).tolist()
    idx = {l:i for i,l in enumerate(labels)}
    link_color = ['rgba(192,57,43,0.5)' if c=='non_technical' else 'rgba(39,174,96,0.4)'
                  for c in edges['routed_queue_category']]
    fig = go.Figure(go.Sankey(
        node=dict(label=labels, pad=15, thickness=14),
        link=dict(source=edges['source_intent'].map(idx),
                  target=edges['target_queue'].map(idx),
                  value=edges['n_sessions'], color=link_color)))
    fig.update_layout(title_text='Technical intent -> destination (red = non-technical queue)',
                      font_size=11, height=600)
    fig.show()
except ImportError:
    print('plotly not installed; showing edge table instead')
    display(edges)

## 8. Topic -> destination confusion heatmap

In [ ]:
cm = run_query(f'''
SELECT technical_topic_type, routed_queue_subtype, COUNT(*) n
FROM {MASTER}
WHERE is_technical_topic AND was_transferred
GROUP BY 1,2
''')
pivot = cm.pivot_table(index='technical_topic_type', columns='routed_queue_subtype',
                       values='n', fill_value=0)
plt.figure(figsize=(10,4))
sns.heatmap(pivot, annot=True, fmt='d', cmap='Reds')
plt.title('Technical topic vs destination queue'); plt.tight_layout(); plt.show()
pivot

## 9. Temporal heatmap (day-of-week x hour)

In [ ]:
hm = run_query(f'''
SELECT EXTRACT(DAYOFWEEK FROM START_MOMENT) dow,
       EXTRACT(HOUR FROM START_MOMENT) hour,
       COUNTIF(is_hard_misroute) misroutes
FROM {MASTER} GROUP BY dow, hour
''')
pivot = hm.pivot_table(index='dow', columns='hour', values='misroutes', fill_value=0)
plt.figure(figsize=(14,4))
sns.heatmap(pivot, cmap='magma')
plt.title('Hard misroutes by day-of-week (1=Sun) x hour'); plt.tight_layout(); plt.show()

## 10. Opportunity sizing (Deliverables 4 & 5)

If misrouted technical sessions were routed correctly, estimated time saved
and avoidable repeat contacts.

In [ ]:
mis = tech[tech['is_hard_misroute']]
cor = tech[tech['is_correct_technical_route']]
excess_s = mis['duration_seconds'].mean() - cor['duration_seconds'].mean()
opp = {
    'misrouted_sessions': len(mis),
    'excess_seconds_per_session': round(excess_s, 1),
    'est_total_hours_saved': round(excess_s*len(mis)/3600, 1),
    'repeat_contacts_avoidable': int(mis['repeat_contact_24h'].sum()),
}
pd.Series(opp).to_frame('value')

## 11. Prioritised fix list

Top intent -> wrong-destination leaks ranked by an impact score
(volume + 2x repeat contacts + handovers). These are the highest-ROI changes.

In [ ]:
leaks = (mis.groupby(['FIRST_INTENT','technical_topic_type','final_transfer_target'])
            .agg(misrouted=('SESSION_ID','count'),
                 repeats=('repeat_contact_24h','sum'),
                 handovers=('n_transfers','sum'))
            .reset_index())
leaks['impact_score'] = leaks['misrouted'] + 2*leaks['repeats'] + leaks['handovers']
leaks.sort_values('impact_score', ascending=False).head(20)

---
*Numbers depend on the queue classification (`models/02`) and technical-topic
keywords (`models/03`). Validate those against the real `T_`/`R_`/`M_`
vocabulary before circulating results.*